# Modèle de langage et génération de séquence

Dans ces travaux pratiques, nous allons étudier les différentes manières de décoder du texte à partir d'un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage : traduction, question/réponse, génération…

Nous prendrons l'exemple de la génération de texte avec GPT-2, un transformer de type *décodeur seul* : les grands modèles de langage actuels (GPT-4, Llama, Mistral…) reposent sur le même principe et se décodent avec les mêmes méthodes.

Nous partirons de la méthode la plus simple pour progresser jusqu'aux méthodes utilisées en pratique, en codant nous-mêmes le choix du token suivant. Cela vous permettra de bien comprendre le rôle de la softmax en sortie des modèles de langage. Pour finir, nous retrouverons tous ces réglages dans la méthode `generate` de KerasHub.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de Hugging Face peut servir de ressource complémentaire à ce notebook.

Ce TP n'a pas besoin de carte graphique : GPT-2 small tourne sur CPU. La boucle de décodage que nous écrirons recalcule toute la séquence à chaque token : sur CPU, chaque test sur les 7 prompts du TP prend une à deux minutes.

**Attention** : ces modèles ont été entraînés sur des textes du web, sans filtrage. Les textes générés, surtout par échantillonnage, peuvent être incohérents, faux ou choquants.

## Installation

La bibliothèque [KerasHub](https://keras.io/keras_hub/) met à disposition des modèles pré-entraînés pour Keras, dont GPT-2. Elle est préinstallée sur Colab ; la cellule suivante l'installe ailleurs (Codespace, poste personnel).

In [ ]:
!pip install -q keras-hub

## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles pré-entraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste du TP.

In [ ]:
lang = "fr"
# lang = "en"

## Import d'un modèle entraîné

Un modèle de KerasHub se charge avec la méthode `from_preset` de sa classe, `keras_hub.models.GPT2CausalLM` pour un GPT-2 qui prédit le token suivant (*causal language model*) : elle récupère l'architecture, les poids et le tokeniseur.

- En anglais, `"gpt2_base_en"` est le GPT-2 small d'OpenAI (2019), publié par KerasHub sur [Kaggle](https://www.kaggle.com/models/keras/gpt2).
- En français, [BelGPT-2](https://huggingface.co/antoinelouis/belgpt2) est un GPT-2 small entraîné sur un corpus français. KerasHub le charge depuis le Hub Hugging Face avec le préfixe `hf://`. Son vocabulaire nomme `</s>` le symbole de fin de texte que KerasHub attend sous le nom `<|endoftext|>` : la cellule construit donc le tokeniseur elle-même, après avoir renommé ce symbole.

Le tokeniseur du modèle se retrouve ensuite dans `causal_lm.preprocessor.tokenizer`.

In [ ]:
import functools
import json
import statistics

import keras
import keras_hub
import tqdm.auto
from huggingface_hub import hf_hub_download

# Mêmes tirages aléatoires à chaque exécution du notebook
keras.utils.set_random_seed(0)

if lang == "fr":
  repo = "antoinelouis/belgpt2"
  with open(hf_hub_download(repo, "vocab.json"), encoding="utf-8") as file:
    vocabulary = json.load(file)
  # Le symbole de fin de texte, sous le nom que KerasHub attend
  vocabulary["<|endoftext|>"] = vocabulary.pop("</s>")
  tokenizer = keras_hub.models.GPT2Tokenizer(
    vocabulary, hf_hub_download(repo, "merges.txt")
  )
  causal_lm = keras_hub.models.GPT2CausalLM.from_preset(
    f"hf://{repo}",
    preprocessor=keras_hub.models.GPT2CausalLMPreprocessor(tokenizer),
  )
else:
  causal_lm = keras_hub.models.GPT2CausalLM.from_preset("gpt2_base_en")

tokenizer = causal_lm.preprocessor.tokenizer

## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques. La cellule suivante affiche le résumé du modèle, celui de son *backbone* (le transformer, sans la couche de sortie), la configuration du backbone et celle de son premier bloc.

- *Combien le modèle a-t-il de couches de transformer ?*
- *Quels sont ses mécanismes de régularisation ?*
- *De stabilisation de l'apprentissage ?*
- *Combien de paramètres possède-t-il ?*
- *Nous aurons besoin plus tard de l'index du symbole spécial qui délimite le début et la fin des textes. Quel est-il ? Le tokeniseur `tokenizer` le donne.*

In [ ]:
causal_lm.summary()
causal_lm.backbone.summary()
print(causal_lm.backbone.get_config())
print(causal_lm.backbone.get_layer("transformer_layer_0").get_config())

*Votre réponse à compléter ici.*
-
-
-
-
-

### Solution

In [ ]:
config = causal_lm.backbone.get_config()
print(f"Couches de transformer : {config['num_layers']}")
print(f"Dropout : {config['dropout']}")
print(f"{causal_lm.count_params() / 1e6:.0f} millions de paramètres")
print(f"Début/fin de texte : {tokenizer.start_token_id} / {tokenizer.end_token_id}")

- Il possède 12 couches de transformer (`num_layers`, ou les blocs `transformer_layer_0` à `transformer_layer_11` du résumé).
- Son mécanisme de régularisation est le dropout, de taux 0,1 (`dropout`, la couche `embeddings_dropout` et le dropout de chaque bloc).
- Ses mécanismes de stabilisation de l'apprentissage sont la normalisation de couche (la couche `layer_norm` en sortie, et dans chaque bloc `layer_norm_epsilon` et `normalize_first` : une normalisation avant l'attention et avant la couche dense) et les connexions résiduelles autour de l'attention et de la couche dense de chaque bloc (internes à `TransformerDecoder`, elles n'apparaissent pas dans le résumé).
- Il a environ 124 millions de paramètres. La couche de sortie `token_embedding (ReversibleEmbedding)` n'en ajoute aucun : elle réutilise la matrice d'embedding des tokens de l'entrée.
- `tokenizer.start_token_id` et `tokenizer.end_token_id` donnent l'index du symbole de début et de fin de texte : 50256 (`<|endoftext|>`) pour GPT-2, 2 pour BelGPT-2.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on procède de manière itérative : on génère les tokens un par un, en commençant par un texte vide ou par une phrase que l'on souhaite compléter.

Pour générer un token, on donne en entrée du réseau les tokens générés jusque-là (sous forme d'indices dans le vocabulaire). Le réseau produit alors un score par token du vocabulaire. On choisit l'un de ces tokens, on l'ajoute à ce qui a déjà été généré, puis on recommence pour générer le token suivant.

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

Tout l'enjeu du décodage est de trouver le texte qui maximise ce produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approcher.

C'est le **masque causal** du transformer qui rend cette décomposition possible : pendant l'entraînement, chaque position ne voit que les positions précédentes.

## Tokenisation

Le tokeniseur récupéré avec le modèle transforme une phrase en indices de vocabulaire (des *sous-mots*) que le modèle comprend. Dans l'affichage des tokens, `Ġ` marque un token qui commence par une espace.

Nous utiliserons dans la suite du TP les fonctions `encode` et `decode` suivantes. `encode` ajoute le symbole de début de texte, et renvoie un tenseur de forme `(1, nombre de tokens)`.

BelGPT-2 a été entraîné sur des textes où la ponctuation est séparée des mots par des espaces (« j' ai », « fin . ») : ses générations en gardent la trace.

In [ ]:
def encode(sentence: str):
  """Indices des tokens, après le symbole de début : forme (1, nombre de tokens)."""
  token_ids = keras.ops.convert_to_tensor(tokenizer(sentence), dtype="int32")
  start = keras.ops.convert_to_tensor([tokenizer.start_token_id], dtype="int32")
  return keras.ops.expand_dims(keras.ops.concatenate([start, token_ids]), axis=0)


def decode(token_ids) -> str:
  """Texte des tokens, sans le symbole de début."""
  return tokenizer.detokenize(token_ids[0, 1:])


example = encode(
  "Je pense, donc je suis" if lang == "fr" else "I think, therefore I am"
)
print(example)
print([tokenizer.id_to_token(int(i)) for i in keras.ops.convert_to_numpy(example[0])])

## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain token. Étudions son comportement.

Comme tout modèle Keras, `causal_lm` s'appelle comme une fonction. Il prend un dictionnaire de deux tenseurs de forme `(batch, nombre de tokens)` :

- `"token_ids"` : les indices des tokens ;
- `"padding_mask"` : 1 pour un vrai token, 0 pour une position de remplissage (ici, nous n'avons que des 1).

Il renvoie les **logits** : les scores du modèle, avant softmax.

*Appliquez le modèle à diverses séquences, par exemple :*

- *« » (le texte vide)*
- *« Je pense, donc je suis » ou « I think, therefore I am » en fonction de la langue du modèle*

*Quelle est la forme des logits ? À quoi correspond chaque dimension ?*

**Documentation utile**

- [`keras_hub.models.GPT2CausalLM`](https://keras.io/keras_hub/api/models/gpt2/gpt2_causal_lm/#gpt2causallm-class)
- [`keras.ops.ones_like`](https://keras.io/api/ops/numpy/#oneslike-function)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `encode(sentence)` donne les `token_ids`, de forme `(1, nombre de tokens)`
- `keras.ops.ones_like(token_ids)` donne le `padding_mask` : un tenseur de 1 de même forme
- `logits.shape` donne la forme des logits
- Résultat attendu pour le texte vide : `(1, 1, 50257)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def describe_shapes(sentence: str) -> None:
  token_ids = encode(sentence)
  logits = causal_lm({"token_ids": ..., "padding_mask": ...})
  print(f"Forme des logits pour '{sentence}' : {...}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")
```

### Solution

In [ ]:
def describe_shapes(sentence: str) -> None:
  token_ids = encode(sentence)
  logits = causal_lm(
    {"token_ids": token_ids, "padding_mask": keras.ops.ones_like(token_ids)}
  )
  print(f"Forme des logits pour '{sentence}' : {tuple(logits.shape)}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")

Les logits ont la forme `(batch, nombre de tokens, taille du vocabulaire)` : pour chaque position de l'entrée, le modèle donne un score à chacun des 50 257 tokens du vocabulaire pour la position **suivante**. Le texte vide a une position, celle du symbole de début qu'`encode` ajoute (Keras avertit alors que la softmax de l'attention porte sur une seule position : c'est attendu). Pour générer, seule la dernière position nous intéresse.

## Fonction `forward` adaptée aux besoins du décodage

Nous utiliserons la fonction `forward` suivante, qui rend un résultat adapté au décodage : seule la dernière position nous intéresse (la prédiction du token suivant).

Elle recalcule toute la séquence à chaque appel : c'est simple, mais coûteux. La méthode `generate` de KerasHub, en fin de TP, garde en mémoire (dans un *cache*) les clés et les valeurs d'attention des tokens déjà vus, et ne calcule que le nouveau token à chaque pas.

In [ ]:
def forward(token_ids):
  """Logits du token suivant : forme (1, taille du vocabulaire)."""
  logits = causal_lm(
    {"token_ids": token_ids, "padding_mask": keras.ops.ones_like(token_ids)}
  )
  return logits[:, -1, :]

`forward` prend une séquence de forme `(1, 7)` par exemple, et renvoie des logits de forme `(1, taille du vocabulaire)` : la dimension du batch, de taille 1, est conservée.

*Pourquoi est-il souvent utile de conserver des dimensions à `1` dans les réseaux de neurones ?*

*Votre réponse à compléter ici.*

### Solution

Ces dimensions permettent de calculer en parallèle des résultats pour plusieurs exemples (batching) : le même code fonctionne pour un batch de 1 comme de 64.

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Passe forward sur tout ce qui a été produit jusqu'ici (entrées et sorties précédentes)
2. Choix de l'indice du token suivant
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (nous utiliserons seulement le nombre de tokens produits dans ce TP)

L'étape 2 est celle où tout se joue. Le code suivant implémente tout le reste, afin de pouvoir nous concentrer sur l'étape 2 : chaque méthode de décodage sera une fonction qui reçoit les logits du dernier token (forme `(1, taille du vocabulaire)`) et renvoie l'indice du token choisi (un tenseur de forme `()`). Décorée par `@decoding_loop`, elle devient une fonction `(prompt, nombre de tokens à générer, paramètres de la méthode)` qui renvoie le texte généré.

In [ ]:
def decoding_loop(step_function):

  @functools.wraps(step_function)
  def wrapper(prompt: str, length: int, *step_args, **step_kwargs) -> str:
    token_ids = encode(prompt)
    for _ in tqdm.auto.trange(length, desc="Tokens", leave=False):
      index = step_function(forward(token_ids), *step_args, **step_kwargs)
      index = keras.ops.cast(keras.ops.reshape(index, (1, 1)), "int32")
      token_ids = keras.ops.concatenate([token_ids, index], axis=1)
    return decode(token_ids)

  return wrapper

## Décodage glouton (*greedy*)

Le décodage glouton est la méthode la plus simple pour décoder du texte depuis un modèle appris : à chaque étape, on prend le token de plus forte probabilité.

Dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

le décodage glouton choisit $\text{Token}_i$ pour maximiser $P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un token moins probable pour ensuite atteindre des tokens très probables.

L'exemple suivant est tiré de l'article de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

Après « The », on choisit « nice » parce que c'est le mot le plus probable, alors que le texte complet « The nice woman » a une probabilité (0,20) inférieure à celle qu'on aurait obtenue en choisissant « dog » (« The dog has », 0,36).

*Codez la fonction `greedy(logits)` qui prend en entrée les logits du modèle et renvoie l'indice du token choisi, un tenseur de forme `()`. Elle utilisera `keras.ops.argmax`. Générez 10 tokens après « Je suis allé au » (« I went to the » en anglais).*

**Documentation utile**

- [`keras.ops.argmax`](https://keras.io/api/ops/numpy/#argmax-function)

In [ ]:
@decoding_loop
def greedy(logits): ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.ops.argmax(logits, axis=-1)` : l'indice du plus grand logit de chaque ligne, de forme `(1,)` ; `[0]` en extrait le tenseur de forme `()`
- Inutile de passer par la softmax : elle ne change pas l'ordre des scores
- `greedy(prompt, 10)` : la fonction décorée prend le prompt et le nombre de tokens à générer

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def greedy(logits):
  return ...


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))
```

### Solution

In [ ]:
@decoding_loop
def greedy(logits):
  return keras.ops.argmax(logits, axis=-1)[0]


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))

## Test sur des prompts variés

La fonction `test_decoding` suivante applique une méthode de décodage, avec ses paramètres, à chacun des prompts de la liste `inputs`. Elle affiche les générations et les renvoie dans une liste ; `show` les affiche.

*Exécutez les cellules suivantes, qui testent le décodage glouton avec 30 tokens par prompt. Que constatez-vous ?*

In [ ]:
if lang == "fr":
  inputs = [
    "Je suis allé au",
    "Le train pour Lyon",
    "Comment vas-tu",
    "La recette de la tarte aux pommes commence par",
    "Ça va merci, tu devrais",
    "Le chat de la voisine",
    "En 2050, les voitures",
  ]
else:
  inputs = [
    "I went to the",
    "The train to London",
    "How do you",
    "The apple pie recipe starts with",
    "I'm fine thank you, you should",
    "The neighbour's cat",
    "In 2050, cars",
  ]

In [ ]:
def show(generations: list[str]) -> None:
  """Affiche chaque prompt de `inputs` avec sa génération."""
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)


def test_decoding(function, *args, **kwargs) -> list[str]:
  """Applique la méthode de décodage `function` à chaque prompt de `inputs`."""
  generations = [
    function(prompt, *args, **kwargs)
    for prompt in tqdm.auto.tqdm(inputs, desc="Prompts", leave=False)
  ]
  show(generations)
  return generations

In [ ]:
greedy_generations = test_decoding(greedy, 30)

## Mesurer la répétition

Les générations gloutonnes « bouclent » vite : elles répètent les mêmes groupes de mots. Pour comparer les méthodes de décodage sur ce point, mesurons-le par la **part de 3-grammes distincts** d'un texte : on découpe le texte en mots, on forme tous les triplets de mots consécutifs (les 3-grammes), et on divise le nombre de triplets différents par le nombre total de triplets. Elle vaut 1 pour un texte sans répétition, et baisse quand des passages se répètent.

*Codez la fonction `distinct_3grams(text)` qui renvoie la part de 3-grammes distincts de `text`. Affichez ensuite sa moyenne sur les générations gloutonnes, `greedy_generations`, avec `statistics.mean`. Les sections suivantes la réutilisent pour chaque méthode.*

In [ ]:
def distinct_3grams(text: str) -> float: ...


# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `text.split()` : la liste des mots, séparés par les espaces
- `zip(words, words[1:], words[2:])` : les triplets de mots consécutifs
- `set(...)` : les triplets différents
- `statistics.mean(map(distinct_3grams, greedy_generations))` : la moyenne sur les générations
- Exemple à vérifier : `"a b c a b c"` a 4 triplets, dont 3 différents : 0,75

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(...)
  return ... / ...


print(distinct_3grams("a b c a b c"))
print(f"Glouton : {statistics.mean(...):.2f}")
```

### Solution

In [ ]:
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(zip(words, words[1:], words[2:]))
  return len(set(trigrams)) / len(trigrams)


print(distinct_3grams("a b c a b c"))
print(f"Glouton : {statistics.mean(map(distinct_3grams, greedy_generations)):.2f}")

Plusieurs générations gloutonnes répètent la même phrase, ou le même groupe de mots, jusqu'à la fin : c'est une faiblesse très classique du décodage glouton. La part de 3-grammes distincts le chiffre : elle est nettement sous 1, là où un texte écrit par un humain en est proche.

## Échantillonnage (*sampling*)

Les résultats du décodage glouton bouclent rapidement et sont souvent très génériques. Les méthodes suivantes cherchent à corriger ces défauts.

La première consiste à **échantillonner** à partir des probabilités du modèle, plutôt que de toujours choisir le token le plus probable : un token de probabilité 0,3 est tiré 3 fois sur 10.

*Reprenez la forme de la fonction `greedy`, mais au lieu de choisir l'élément maximal, tirez le prochain token avec `keras.random.categorical` dans une fonction `sampling(logits)`. Testez-la avec `test_decoding` (30 tokens par prompt), et comparez sa part de 3-grammes distincts à celle du décodage glouton.*

**Documentation utile**

- [`keras.random.categorical`](https://keras.io/api/random/random_ops/#categorical-function)

In [ ]:
@decoding_loop
def sampling(logits): ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.random.categorical(logits, 1)` : un tirage par ligne de `logits`, selon les probabilités `softmax(logits)` ; la softmax est faite par la fonction, qui prend directement les logits
- Le résultat a la forme `(1, 1)` : `[0, 0]` en extrait le tenseur de forme `()`
- `statistics.mean(map(distinct_3grams, generations))` : la part de 3-grammes distincts

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def sampling(logits):
  return ...


generations = test_decoding(sampling, 30)
print(f"3-grammes distincts : {...:.2f}")
```

### Solution

In [ ]:
@decoding_loop
def sampling(logits):
  return keras.random.categorical(logits, 1)[0, 0]


generations = test_decoding(sampling, 30)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")

Les générations ne bouclent plus : la part de 3-grammes distincts est proche de 1. En contrepartie, elles perdent souvent le fil, parce qu'un token peu probable est tiré de temps en temps et que la suite du texte doit s'en accommoder.

## Température

Avant la softmax, on peut diviser les logits par une **température** $T$ :

- $T < 1$ accentue les écarts : le modèle choisit plus souvent les tokens les plus probables (plus cohérent, moins varié) ;
- $T > 1$ les aplatit : les tokens rares sont plus souvent tirés (plus varié, moins cohérent) ;
- $T \to 0$ revient au décodage glouton.

*Ajoutez un paramètre `temperature` à votre fonction d'échantillonnage, dans une nouvelle fonction `temperature_sampling(logits, temperature)`. Comparez les générations et leur part de 3-grammes distincts pour $T = 0.5$ et $T = 1.5$.*

In [ ]:
@decoding_loop
def temperature_sampling(logits, temperature: float): ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `logits / temperature` : les logits divisés par la température, avant le tirage
- `temperature_sampling(prompt, 30, temperature=0.5)` : la fonction décorée transmet ses paramètres nommés à la vôtre ; `test_decoding` aussi

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def temperature_sampling(logits, temperature: float):
  return keras.random.categorical(..., 1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 30, temperature=temperature)
  print(f"T = {temperature} : {...:.2f} de 3-grammes distincts")
```

### Solution

In [ ]:
@decoding_loop
def temperature_sampling(logits, temperature: float):
  return keras.random.categorical(logits / temperature, 1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 30, temperature=temperature)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"T = {temperature} : {repetition:.2f} de 3-grammes distincts")

À $T = 0.5$, les générations sont plus cohérentes, mais se rapprochent du décodage glouton et peuvent recommencer à boucler. À $T = 1.5$, elles n'ont presque plus de sens : des tokens rares sont tirés à chaque pas. La température règle ce compromis entre cohérence et variété.

## Top-k sampling

Une variation de l'échantillonnage consiste à ne tirer que parmi les `k` tokens les plus probables, pour éviter de générer un token très improbable.

*Repartez de la fonction `sampling` pour coder la fonction `k_sampling(logits, k)` qui implémente cette amélioration avec `keras.ops.top_k`. Testez-la avec `test_decoding`, pour `k = 20`.*

**Documentation utile**

- [`keras.ops.top_k`](https://keras.io/api/ops/core/#topk-function)

In [ ]:
@decoding_loop
def k_sampling(logits, k: int): ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `values, indices = keras.ops.top_k(logits[0], k=k)` : les `k` plus grands logits, par ordre décroissant, et leurs indices dans le vocabulaire
- `keras.random.categorical(values[None, :], 1)[0, 0]` : tire une position parmi les `k` (`[None, :]` rajoute la dimension du batch)
- `indices[position]` : l'indice dans le vocabulaire du token tiré

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def k_sampling(logits, k: int):
  values, indices = keras.ops.top_k(...)
  sampled = keras.random.categorical(..., 1)[0, 0]
  return indices[...]


generations = test_decoding(k_sampling, 30, k=20)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")
```

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits, k: int):
  values, indices = keras.ops.top_k(logits[0], k=k)
  sampled = keras.random.categorical(values[None, :], 1)[0, 0]
  return indices[sampled]


generations = test_decoding(k_sampling, 30, k=20)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")

## Top-p sampling

Une autre amélioration consiste à ne garder que les tokens les plus probables dont la probabilité cumulée atteint `p` (le *noyau* de la distribution), et pas les suivants. Le nombre de tokens candidats s'adapte alors à la confiance du modèle : peu de candidats quand un token domine, beaucoup quand la distribution est plate.

*Repartez de la fonction `k_sampling` et ajoutez-y le top-p sampling, dans la fonction `k_p_sampling(logits, k, p)`. Vous pourrez utiliser `keras.ops.softmax` et `keras.ops.cumsum`. Testez-la avec `test_decoding`, pour `k = 20` et `p = 0.85`.*

**Documentation utile**

- [`keras.ops.softmax`](https://keras.io/api/ops/nn/#softmax-function)
- [`keras.ops.cumsum`](https://keras.io/api/ops/numpy/#cumsum-function)

In [ ]:
@decoding_loop
def k_p_sampling(logits, k: int, p: float): ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.ops.softmax(values)` : les probabilités des `k` tokens gardés, par ordre décroissant
- `keras.ops.cumsum(probabilities) < p` : vrai pour les tokens avant le seuil ; `keras.ops.sum` les compte, et on ajoute 1 pour garder le token qui franchit `p`
- `int(...)` convertit ce nombre en entier Python, pour s'en servir dans une tranche
- Exemple à vérifier : pour les probabilités `[0.5, 0.3, 0.15, 0.05]` et `p = 0.85`, les sommes cumulées sont `[0.5, 0.8, 0.95, 1.0]` : 2 sont sous `p`, on garde 3 tokens
- `keras.random.categorical(values[None, :n_kept], 1)` tire parmi les `n_kept` premiers

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def k_p_sampling(logits, k: int, p: float):
  values, indices = keras.ops.top_k(logits[0], k=k)
  probabilities = ...
  n_kept = int(keras.ops.sum(...)) + 1
  sampled = keras.random.categorical(..., 1)[0, 0]
  return indices[sampled]


generations = test_decoding(k_p_sampling, 30, k=20, p=0.85)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")
```

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits, k: int, p: float):
  values, indices = keras.ops.top_k(logits[0], k=k)
  probabilities = keras.ops.softmax(values)
  # Nombre de tokens nécessaires pour que la probabilité cumulée atteigne p
  n_kept = int(keras.ops.sum(keras.ops.cumsum(probabilities) < p)) + 1
  sampled = keras.random.categorical(values[None, :n_kept], 1)[0, 0]
  return indices[sampled]


generations = test_decoding(k_p_sampling, 30, k=20, p=0.85)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")

## Génération avec `generate`

En pratique, on n'écrit pas la boucle de décodage : la méthode `generate` des modèles de KerasHub implémente toutes ces stratégies. Elle garde en cache les clés et les valeurs d'attention, et compile la boucle avec XLA : la première génération prend une trentaine de secondes de compilation, les suivantes sont rapides. Le choix du token suivant se règle par `causal_lm.compile(sampler=...)`, avec les échantillonneurs de `keras_hub.samplers` : `GreedySampler`, `RandomSampler`, `TopKSampler`, `TopPSampler`, qui ont tous un paramètre `temperature`.

*Générez une centaine de tokens pour chaque prompt de `inputs`, en un seul appel de `generate` sur la liste : `max_length=110` compte les tokens du prompt. Faites-le d'abord avec un top-p sampling restreint aux 30 tokens les plus probables, de noyau `p = 0.95`, puis avec le décodage glouton. Affichez les générations avec `show`, et comparez leur part de 3-grammes distincts.*

**Documentation utile**

- [`GPT2CausalLM.generate`](https://keras.io/keras_hub/api/models/gpt2/gpt2_causal_lm/#generate-method)
- [`keras_hub.samplers.TopPSampler`](https://keras.io/keras_hub/api/samplers/top_p_sampler/#toppsampler-class)
- [`keras_hub.samplers.GreedySampler`](https://keras.io/keras_hub/api/samplers/greedy_sampler/#greedysampler-class)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras_hub.samplers.TopPSampler(p=0.95, k=30)` : le top-p sampling, restreint aux `k` tokens les plus probables, comme `k_p_sampling`
- `keras_hub.samplers.GreedySampler()` : le décodage glouton
- `causal_lm.compile(sampler=sampler)` change l'échantillonneur de `generate`
- `causal_lm.generate(inputs, max_length=110)` : une liste de textes, prompts compris, dans l'ordre de `inputs`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for sampler in [..., ...]:
  causal_lm.compile(sampler=sampler)
  generations = causal_lm.generate(...)
  show(generations)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"{type(sampler).__name__} : {repetition:.2f} de 3-grammes distincts")
```

### Solution

In [ ]:
samplers = [
  keras_hub.samplers.TopPSampler(p=0.95, k=30),
  keras_hub.samplers.GreedySampler(),
]
for sampler in samplers:
  causal_lm.compile(sampler=sampler)
  generations = causal_lm.generate(inputs, max_length=110)
  show(generations)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"{type(sampler).__name__} : {repetition:.2f} de 3-grammes distincts")

Sur une centaine de tokens, l'écart se creuse : la plupart des générations gloutonnes finissent par boucler, et leur part de 3-grammes distincts chute, alors que le top-p sampling reste varié. C'est pourquoi les modèles de langage sont presque toujours décodés par échantillonnage, avec une température, un top-k et un top-p réglés pour l'usage visé.